In [20]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

from tensorboardX import SummaryWriter      

import torchvision
import torchvision.transforms as transforms

import torch.nn.utils.prune as prune

from models import *

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128
model_name = "VGG16_quant"
model = VGG16_quant()
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

In [2]:
# HW

#  1. Load your saved model and validate
#  2. Replace your model's all the Conv's weight with quantized weight
#  3. Apply reasonable alpha
#  4. Then, try to multiple bit precisions and draw graph of bit precision vs. accuracy

In [3]:
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 9279/10000 (93%)



In [4]:
#### Prune all the QuantConv2D layers' 80% weights with 1) unstructured, and 2) structured manner.

In [5]:
i = 0
for layer in model.modules():
    i = i+1
    if isinstance(layer, QuantConv2d):
        if not isinstance(layer.weight, nn.Parameter):
            layer.weight = nn.Parameter(layer.weight.data)
        prune.random_unstructured(layer, name="weight", amount=0.8)

In [6]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_org, ...
print(model.features[40].weight) # check whether there are many zeros

[('act_alpha', Parameter containing:
tensor(0.6882, device='cuda:0', requires_grad=True)), ('weight_q', Parameter containing:
tensor([[[[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         ...,

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]]],


        [[[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.5025, -0.0000],
          [ 0.0000,  0.5025, -0.0000]],

         [[-0.5025, -0.5

In [7]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)

Sparsity level:  tensor(0.8000, device='cuda:0')


In [8]:
model.cuda()
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 1000/10000 (10%)



In [9]:
lr = 1e-1
weight_decay = 1e-4
epochs = 50
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+'pruning'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.290 (0.290)	Data 0.101 (0.101)	Loss 6.5203 (6.5203)	Prec 12.500% (12.500%)
Epoch: [0][100/391]	Time 0.056 (0.058)	Data 0.002 (0.003)	Loss 1.6636 (1.8990)	Prec 37.500% (29.146%)
Epoch: [0][200/391]	Time 0.056 (0.057)	Data 0.002 (0.002)	Loss 1.3912 (1.7235)	Prec 51.562% (35.708%)
Epoch: [0][300/391]	Time 0.056 (0.057)	Data 0.002 (0.002)	Loss 1.1031 (1.6046)	Prec 57.031% (40.532%)
Validation starts
Test: [0/79]	Time 0.117 (0.117)	Loss 1.1118 (1.1118)	Prec 57.812% (57.812%)
 * Prec 55.370% 
best acc: 55.370000
Epoch: [1][0/391]	Time 0.176 (0.176)	Data 0.135 (0.135)	Loss 1.2195 (1.2195)	Prec 56.250% (56.250%)
Epoch: [1][100/391]	Time 0.056 (0.057)	Data 0.001 (0.003)	Loss 1.1488 (1.1519)	Prec 60.156% (59.011%)
Epoch: [1][200/391]	Time 0.056 (0.056)	Data 0.002 (0.002)	Loss 1.3266 (1.1310)	Prec 50.000% (59.340%)
Epoch: [1][300/391]	Time 0.056 (0.056)	Data 0.002 (0.002)	Loss 1.0105 (1.0981)	Prec 60.938% (60.538%)
Validation starts
Test: [0/79]	Time 0.125 (0.125)	Loss 1.

In [10]:
PATH = "result/pruningVGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict']) 

model.cuda()

model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8559/10000 (86%)



In [11]:
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model = VGG16_quant()

model.load_state_dict(checkpoint['state_dict'])
model.cuda()
i = 0
for layer in model.modules():
    i = i+1
    if isinstance(layer, QuantConv2d):
        prune.ln_structured(layer, name="weight", amount=0.8, n = 1, dim = 0)

In [12]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_org, ...
print(model.features[40].weight) # check whether there are many zeros

[('act_alpha', Parameter containing:
tensor(0.6882, device='cuda:0', requires_grad=True)), ('weight_q', Parameter containing:
tensor([[[[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         ...,

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]]],


        [[[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.5025, -0.0000],
          [ 0.0000,  0.5025, -0.0000]],

         [[-0.5025, -0.5

In [13]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)

Sparsity level:  tensor(0.8008, device='cuda:0')


In [14]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 1000/10000 (10%)



In [15]:
lr = 1e-1
weight_decay = 1e-4
epochs = 50
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+'pruning'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.163 (0.163)	Data 0.121 (0.121)	Loss 4.7345 (4.7345)	Prec 13.281% (13.281%)
Epoch: [0][100/391]	Time 0.055 (0.057)	Data 0.002 (0.003)	Loss 1.6697 (2.0868)	Prec 37.500% (20.831%)
Epoch: [0][200/391]	Time 0.057 (0.056)	Data 0.001 (0.002)	Loss 1.4265 (1.8738)	Prec 47.656% (28.591%)
Epoch: [0][300/391]	Time 0.058 (0.056)	Data 0.001 (0.002)	Loss 1.4242 (1.7492)	Prec 51.562% (33.646%)
Validation starts
Test: [0/79]	Time 0.111 (0.111)	Loss 1.6178 (1.6178)	Prec 42.188% (42.188%)
 * Prec 43.470% 
best acc: 43.470000
Epoch: [1][0/391]	Time 0.185 (0.185)	Data 0.143 (0.143)	Loss 1.4418 (1.4418)	Prec 44.531% (44.531%)
Epoch: [1][100/391]	Time 0.056 (0.057)	Data 0.001 (0.003)	Loss 1.3692 (1.3983)	Prec 42.969% (47.842%)
Epoch: [1][200/391]	Time 0.063 (0.056)	Data 0.001 (0.002)	Loss 1.3772 (1.3732)	Prec 48.438% (49.192%)
Epoch: [1][300/391]	Time 0.056 (0.056)	Data 0.001 (0.002)	Loss 1.2291 (1.3511)	Prec 55.469% (50.117%)
Validation starts
Test: [0/79]	Time 0.109 (0.109)	Loss 1.

In [26]:
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH, map_location="cuda")
model = VGG16_quant()
model.load_state_dict(checkpoint['state_dict'])
model.cuda()
model.eval()

structured_amount = 0.5 
final_target_sparsity = 0.8

params_to_prune = []

for layer in model.modules():
    if isinstance(layer, QuantConv2d):
        prune.ln_structured(
            layer,
            name="weight",
            amount=structured_amount,
            n=1,
            dim=0,
        )
        params_to_prune.append((layer, "weight"))

s1 = structured_amount
S  = final_target_sparsity

if s1 < S:
    extra_unstruct = 1.0 - (1.0 - S) / (1.0 - s1)
else:
    extra_unstruct = 0.0
prune.global_unstructured(
    params_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=extra_unstruct,  # 0.2
)



In [27]:
def total_model_sparsity(model):
    total_elems = 0
    zero_elems = 0

    for m in model.modules():
        if isinstance(m, QuantConv2d):
            w = m.weight.data 
            total_elems += w.numel()
            zero_elems += (w == 0).sum().item()

    return zero_elems / total_elems

print("Model Sparsity =", total_model_sparsity(model))


Model Sparsity = 0.7999999864042358


In [ ]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

In [28]:
lr = 1e-1
weight_decay = 1e-4
epochs = 50
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+'pruning'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.210 (0.210)	Data 0.151 (0.151)	Loss 6.6269 (6.6269)	Prec 5.469% (5.469%)
Epoch: [0][100/391]	Time 0.054 (0.056)	Data 0.002 (0.003)	Loss 2.1510 (2.3960)	Prec 20.312% (13.784%)
Epoch: [0][200/391]	Time 0.056 (0.056)	Data 0.002 (0.003)	Loss 2.2951 (2.3251)	Prec 14.844% (13.685%)
Epoch: [0][300/391]	Time 0.054 (0.056)	Data 0.002 (0.003)	Loss 1.8845 (2.2680)	Prec 17.188% (14.374%)
Validation starts
Test: [0/79]	Time 0.156 (0.156)	Loss 1.9377 (1.9377)	Prec 25.000% (25.000%)
 * Prec 23.580% 
best acc: 23.580000
Epoch: [1][0/391]	Time 0.232 (0.232)	Data 0.181 (0.181)	Loss 1.7867 (1.7867)	Prec 28.125% (28.125%)
Epoch: [1][100/391]	Time 0.055 (0.056)	Data 0.002 (0.004)	Loss 1.7410 (1.7825)	Prec 21.875% (26.709%)
Epoch: [1][200/391]	Time 0.054 (0.055)	Data 0.001 (0.003)	Loss 1.5293 (1.7089)	Prec 41.406% (30.243%)
Epoch: [1][300/391]	Time 0.054 (0.055)	Data 0.002 (0.002)	Loss 1.2974 (1.6076)	Prec 52.344% (35.016%)
Validation starts
Test: [0/79]	Time 0.135 (0.135)	Loss 1.02

KeyboardInterrupt: 

In [ ]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))